In [2]:
import os
os.environ["R_HOME"] = r"C:\\Program Files\\R\\R-4.4.1" 
os.environ["PATH"] = r"C:\\Program Files\\R\\R-4.4.1\\bin\\x64" + ";" + os.environ["R_HOME"] 
import pandas as pd
import numpy as np
import json
import ast
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
import time
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PowerTransformer
from sklearn.naive_bayes import GaussianNB
import numpy as np
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
import pyarrow.feather as feather
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PowerTransformer
from sklearn.naive_bayes import GaussianNB
import numpy as np
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
import pickle




C:\Users\Tibuman\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
from sklearn.discriminant_analysis import StandardScaler


def preprocess(X_train, X_test, y_train, y_test, impute_strategy = 'knn', transformation = 'log', resample_strategy = 'adasyn', scaler = 'standard'):
    # Returns arrays, must be converted to DataFrames!

    if impute_strategy == 'knn' or impute_strategy == 'KNN':
        # Simple (mean) or KNN Imputation
        imputer = KNNImputer(n_neighbors=5)
        X_train_imputed = imputer.fit_transform(X_train)
        X_test_imputed = imputer.transform(X_test)
    elif impute_strategy == 'mean' or impute_strategy == 'simple':
        imputer = SimpleImputer(strategy='mean')
        X_train_imputed = imputer.fit_transform(X_train)
        X_test_imputed = imputer.transform(X_test)
    elif impute_strategy == 'None' or impute_strategy == 'none':
        X_train_imputed = X_train
        X_test_imputed = X_test

    # Treat zeroes
    X_train_imputed[X_train_imputed == 0] = 1e-5
    X_test_imputed[X_test_imputed == 0] = 1e-5

    X_train = X_train_imputed
    X_test = X_test_imputed

    if transformation == 'log' or transformation == 'Log':
        # Ensure all values are positive by shifting data
        shift_constant = abs(X_train.min().min()) + 1  # Shift to make all values positive
        X_train_shifted = X_train + shift_constant
        X_test_shifted = X_test + shift_constant

        # Apply log transformation
        X_train_transformed = np.log1p(X_train_shifted)
        X_test_transformed = np.log1p(X_test_shifted)

    elif transformation == 'Yeo-Johnson' or transformation == 'yeo-johnson':
        transformer = PowerTransformer(method='yeo-johnson')
        X_train_transformed = transformer.fit_transform(X_train)
        X_test_transformed = transformer.transform(X_test)
    elif transformation == 'None' or transformation == 'none':
        X_train_transformed = X_train
        X_test_transformed = X_test

    X_train = X_train_transformed
    X_test = X_test_transformed

    if resample_strategy == 'ADASYN' or resample_strategy == 'adasyn':
        X_train, y_train = ADASYN(random_state=1337).fit_resample(X_train, y_train)

    elif resample_strategy == 'SMOTE' or resample_strategy == 'smote':
        X_train, y_train = SMOTE(random_state=1337).fit_resample(X_train, y_train)
    elif resample_strategy == 'None' or resample_strategy == 'none':
        # No resampling, do nothing
        pass
            

    if scaler == 'None' or scaler == 'none':
        return X_train, X_test, y_train, y_test
    
    
    if scaler == 'standard' or scaler == 'Standard':
        scaler = StandardScaler()
    elif scaler == 'minmax' or scaler == 'MinMax':
        scaler = MinMaxScaler()
    elif scaler == 'robust' or scaler == 'Robust':
        scaler = RobustScaler()
    else:
        raise ValueError(f"Invalid scaler: {scaler}")
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test



def calculate_specificity_sensitivity(y_true, y_pred):
    """
    Calculate specificity and sensitivity from confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    return specificity, sensitivity


def process_all_datasets(base_path, models, impute_strategy = 'knn', transformation = 'log', resample_strategy = 'adasyn', scaler = 'standard'):
    """
    Process all datasets in the SIMON directory.
    """
    datasets = []
    # get the existing datasets by checking the existing directories

    for dataset_number in range(1, 206):  # Assuming 34 datasets
        # if file does not exist, skip
        if not os.path.exists(os.path.join(base_path, str(dataset_number))):
            print(f"Dataset {dataset_number} not found.")
            continue
        try:
            # Load data
            dataset_path = os.path.join(base_path, str(dataset_number))
            data_training = feather.read_table(os.path.join(dataset_path, 'data_training.feather')).to_pandas()
            data_testing = feather.read_table(os.path.join(dataset_path, 'data_testing.feather')).to_pandas()

            print(f"Processing Dataset {dataset_number}...")
            best_model_result = process_single_dataset_optimization(data_training, data_testing, models, impute_strategy, transformation, resample_strategy, scaler)

            if best_model_result:
                # Collect the feature importances for this dataset
                feature_importances = best_model_result.get('feature_importance', None)
                best_model_result['feature_importance'] = feature_importances

                datasets.append((dataset_number, best_model_result))
        except Exception as e:
            print(f"Error processing dataset {dataset_number}: {e}")

    return datasets



def optimize_model(model, X_train, y_train, param_distributions, n_iter=10, cv=5, random_state=42, n_jobs=-1):
    """
    Optimize model hyperparameters using Randomized Search with cross-validation.
    
    Args:
    - model: The machine learning model to optimize.
    - X_train: Training data features.
    - y_train: Training data labels.
    - param_distributions: Dictionary containing hyperparameter distributions for RandomizedSearchCV.
    - n_iter: Number of parameter settings to sample.
    - cv: Number of cross-validation splits.
    - random_state: Random seed for reproducibility.
    - n_jobs: Number of CPU cores to use for parallel computation.

    Returns:
    - best_model: The best model found after optimization.
    - best_params: The best parameters found by RandomizedSearchCV.
    """

    def custom_scorer(y_true, y_pred_proba):
    # Calculate PR AUC
        pr_auc = average_precision_score(y_true, y_pred_proba)
        
        y_pred_class = (y_pred_proba >= 0.5).astype(int)
        
        f1 = f1_score(y_true, y_pred_class)
        
        recall = recall_score(y_true, y_pred_class)
        
        if np.isnan(pr_auc):
            pr_auc = 0
        if np.isnan(f1):
            f1 = 0
        if np.isnan(recall):
            recall = 0
    
        combined_score = 0.5 * recall + 0.3 * pr_auc + 0.2 * f1
        return combined_score

    # Create the custom scorer
    custom_scorer_func = make_scorer(custom_scorer, greater_is_better=True)

    # Initialize RandomizedSearchCV with custom scoring
    randomized_search = RandomizedSearchCV(
        estimator=model, 
        param_distributions=param_distributions, 
        n_iter=n_iter, 
        cv=cv, 
        random_state=random_state, 
        n_jobs=n_jobs, 
        verbose=2, 
        scoring=custom_scorer_func  # Use the custom scorer
    )

    # Perform Randomized Search
    randomized_search.fit(X_train, y_train)

    # Get the best model and the corresponding parameters
    best_model = randomized_search.best_estimator_
    best_params = randomized_search.best_params_

    # Print the best parameters found
    print(f"Best parameters: {best_params}")

    return best_model, best_params


def process_single_dataset_optimization(data_training, data_testing, models, impute_strategy='knn', transformation='log', 
                                         resample_strategy='adasyn', scaler='standard'):
    """
    Process a single dataset: split, train, evaluate, and filter models.
    Returns the best model and its performance metrics.
    """
    results = []
    
    # Map 'outcome' to binary (0: low, 1: high)
    data_training['outcome'] = data_training['outcome'].map({'low': 0, 'high': 1})
    data_testing['outcome'] = data_testing['outcome'].map({'low': 0, 'high': 1})

    if data_training.isnull().any().any():
        print("Missing values detected, filling with median...")
        data_training.fillna(data_training.median(), inplace=True)

    # Split training and test data
    X_train, y_train = data_training.drop(columns=['outcome']), data_training['outcome']
    X_test, y_test = data_testing.drop(columns=['outcome']), data_testing['outcome']

    # Filter datasets with fewer than 8 subjects in the test set
    if len(y_test) < 8:
        print("Dataset skipped due to insufficient test samples (<8).")
        return None

    X_train, X_test, y_train, y_test = preprocess(X_train, X_test, y_train, y_test, impute_strategy, 
                                                  transformation, resample_strategy, scaler)

    # Convert back to DataFrame for compatibility
    X_train = pd.DataFrame(X_train, columns=data_training.drop(columns=['outcome']).columns)
    X_test = pd.DataFrame(X_test, columns=data_testing.drop(columns=['outcome']).columns)

    # Define hyperparameter grids for each model
    param_grids = {
        "Random Forest": {
            'n_estimators': np.arange(100, 500, 100),
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', None]
        },
        "Logistic Regression": {
            'C': np.logspace(-3, 3, 7),
            'penalty': ['l2'],
            'solver': ['lbfgs', 'newton-cg', 'saga']
        },
        "Support Vector Classifier": {
            'C': np.logspace(-3, 3, 7),
            'kernel': ['linear', 'rbf', 'poly'],
            'gamma': ['scale', 'auto']
        },
        "Decision Tree": {
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', None]
        }
    }   

    # Train and evaluate models
    for model_name, model in models.items():
        print(f"\nOptimizing and training {model_name}...")

        # Optimize hyperparameters using grid search
        best_model, best_params = optimize_model(model, X_train, y_train, param_grids[model_name])
        print(f"Best parameters for {model_name}: {best_params}")
        
        # Perform cross-validation
        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        auc_scores = []
        f1_scores = []
        pr_auc_scores = []
        
        for train_idx, val_idx in skf.split(X_train, y_train):
            best_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
            y_val_pred = best_model.predict_proba(X_train.iloc[val_idx])[:, 1]
            y_val_pred_class = best_model.predict(X_train.iloc[val_idx])

            # Calculate AUC, F1 Score, and PR AUC
            auc_scores.append(roc_auc_score(y_train.iloc[val_idx], y_val_pred))
            f1_scores.append(f1_score(y_train.iloc[val_idx], y_val_pred_class))
            pr_auc_scores.append(average_precision_score(y_train.iloc[val_idx], y_val_pred))

        # Train on the entire training set
        best_model.fit(X_train, y_train)

        # Evaluate on test data
        y_test_pred_proba = best_model.predict_proba(X_test)[:, 1]
        y_test_pred_class = best_model.predict(X_test)
        test_auc = roc_auc_score(y_test, y_test_pred_proba)
        f1 = f1_score(y_test, y_test_pred_class)
        pr_auc = average_precision_score(y_test, y_test_pred_proba)
        specificity, sensitivity = calculate_specificity_sensitivity(y_test, y_test_pred_class)

        # Calculate combined score
        combined_score = 0.5 * sensitivity + 0.3 * pr_auc + 0.2 * f1

        # Filter poorly performing models
        if test_auc < 0.4 or (specificity < 0.5 and sensitivity < 0.5):
            print(f"{model_name} discarded due to poor performance (AUROC={test_auc:.2f}).")
            continue

        # Check if the model has feature importances (for tree-based models)
        feature_importance = None
        if hasattr(best_model, 'feature_importances_'):
            feature_importance = best_model.feature_importances_
        elif hasattr(best_model, 'coef_'):  # For linear models
            feature_importance = best_model.coef_[0]  # For binary classification
        else:
            feature_importance = None

        # Save the results, including feature importance if available
        results.append({
            'model': model_name,
            'test_auc': test_auc,
            'f1_score': f1,
            'pr_auc': pr_auc,
            'specificity': specificity,
            'sensitivity': sensitivity,
            'combined_score': combined_score,
            'best_params': best_params,
            'feature_importance': feature_importance,  # Include feature importances
            'conf_matrix': confusion_matrix(y_test, y_test_pred_class),
            'report': classification_report(y_test, y_test_pred_class, output_dict=True)
        })

    # Return the best model by combined score
    if results:
        best_result = max(results, key=lambda x: x['combined_score'])
        print(f"Best model: {best_result['model']} with combined score={best_result['combined_score']:.3f}")
        return best_result
    else:
        print("No suitable models for this dataset.")
        return None

In [22]:
# Work for datasets which yielded models with no available feature importances (part 1)
def process_single_dataset_optimization_guaranteed_feature_importances(data_training, data_testing, models, impute_strategy='knn', transformation='log', 
                                         resample_strategy='adasyn', scaler='standard'):
    """
    Process a single dataset: split, train, evaluate, and filter models.
    Returns the best model and its performance metrics.
    """
    results = []
    
    # Map 'outcome' to binary (0: low, 1: high)
    data_training['outcome'] = data_training['outcome'].map({'low': 0, 'high': 1})
    data_testing['outcome'] = data_testing['outcome'].map({'low': 0, 'high': 1})

    if data_training.isnull().any().any():
        print("Missing values detected, filling with median...")
        data_training.fillna(data_training.median(), inplace=True)

    # Split training and test data
    X_train, y_train = data_training.drop(columns=['outcome']), data_training['outcome']
    X_test, y_test = data_testing.drop(columns=['outcome']), data_testing['outcome']

    # Filter datasets with fewer than 8 subjects in the test set
    if len(y_test) < 8:
        print("Dataset skipped due to insufficient test samples (<8).")
        return None

    X_train, X_test, y_train, y_test = preprocess(X_train, X_test, y_train, y_test, impute_strategy, 
                                                  transformation, resample_strategy, scaler)

    # Convert back to DataFrame for compatibility
    X_train = pd.DataFrame(X_train, columns=data_training.drop(columns=['outcome']).columns)
    X_test = pd.DataFrame(X_test, columns=data_testing.drop(columns=['outcome']).columns)

    # Define hyperparameter grids for each model
    param_grids = {
        "Random Forest": {
            'n_estimators': np.arange(100, 500, 100),
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', None]
        },
        "Logistic Regression": {
            'C': np.logspace(-3, 3, 7),
            'penalty': ['l2'],
            'solver': ['lbfgs', 'newton-cg', 'saga']
        },
        "Support Vector Classifier": {
            'C': np.logspace(-3, 3, 7),
            'kernel': ['linear'],
            'gamma': ['scale', 'auto']
        },
        "Decision Tree": {
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', None]
        }
    }   

    # Train and evaluate models
    for model_name, model in models.items():
        print(f"\nOptimizing and training {model_name}...")

        # Optimize hyperparameters using grid search
        best_model, best_params = optimize_model(model, X_train, y_train, param_grids[model_name])
        print(f"Best parameters for {model_name}: {best_params}")
        
        # Perform cross-validation
        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        auc_scores = []
        f1_scores = []
        pr_auc_scores = []
        
        for train_idx, val_idx in skf.split(X_train, y_train):
            best_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
            y_val_pred = best_model.predict_proba(X_train.iloc[val_idx])[:, 1]
            y_val_pred_class = best_model.predict(X_train.iloc[val_idx])

            # Calculate AUC, F1 Score, and PR AUC
            auc_scores.append(roc_auc_score(y_train.iloc[val_idx], y_val_pred))
            f1_scores.append(f1_score(y_train.iloc[val_idx], y_val_pred_class))
            pr_auc_scores.append(average_precision_score(y_train.iloc[val_idx], y_val_pred))

        # Train on the entire training set
        best_model.fit(X_train, y_train)

        # Evaluate on test data
        y_test_pred_proba = best_model.predict_proba(X_test)[:, 1]
        y_test_pred_class = best_model.predict(X_test)
        test_auc = roc_auc_score(y_test, y_test_pred_proba)
        f1 = f1_score(y_test, y_test_pred_class)
        pr_auc = average_precision_score(y_test, y_test_pred_proba)
        specificity, sensitivity = calculate_specificity_sensitivity(y_test, y_test_pred_class)

        # Calculate combined score
        combined_score = 0.5 * sensitivity + 0.3 * pr_auc + 0.2 * f1

        # Filter poorly performing models
        if test_auc < 0.4 or (specificity < 0.5 and sensitivity < 0.5):
            print(f"{model_name} discarded due to poor performance (AUROC={test_auc:.2f}).")
            continue

        # Check if the model has feature importances (for tree-based models)
        feature_importance = None
        if hasattr(best_model, 'feature_importances_'):
            feature_importance = best_model.feature_importances_
        elif hasattr(best_model, 'coef_'):  # For linear models
            feature_importance = best_model.coef_[0]  # For binary classification
        else:
            feature_importance = None

        # Save the results, including feature importance if available
        results.append({
            'model': model_name,
            'test_auc': test_auc,
            'f1_score': f1,
            'pr_auc': pr_auc,
            'specificity': specificity,
            'sensitivity': sensitivity,
            'combined_score': combined_score,
            'best_params': best_params,
            'feature_importance': feature_importance,  # Include feature importances
            'conf_matrix': confusion_matrix(y_test, y_test_pred_class),
            'report': classification_report(y_test, y_test_pred_class, output_dict=True)
        })

    # Return the best model by combined score
    if results:
        best_result = max(results, key=lambda x: x['combined_score'])
        print(f"Best model: {best_result['model']} with combined score={best_result['combined_score']:.3f}")
        return best_result
    else:
        print("No suitable models for this dataset.")
        return None

def process_some_datasets(base_path, models, impute_strategy = 'knn', transformation = 'log', resample_strategy = 'adasyn', scaler = 'standard', dataset_numbers = [171]):
    """
    Process only some datasets in the SIMON directory.
    """
    datasets = []
    # get the existing datasets by checking the existing directories

    for dataset_number in dataset_numbers:
        # if file does not exist, skip
        if not os.path.exists(os.path.join(base_path, str(dataset_number))):
            print(f"Dataset {dataset_number} not found.")
            continue
        try:
            # Load data
            dataset_path = os.path.join(base_path, str(dataset_number))
            data_training = feather.read_table(os.path.join(dataset_path, 'data_training.feather')).to_pandas()
            data_testing = feather.read_table(os.path.join(dataset_path, 'data_testing.feather')).to_pandas()

            print(f"Processing Dataset {dataset_number}...")
            best_model_result = process_single_dataset_optimization_guaranteed_feature_importances(data_training, data_testing, models, impute_strategy, transformation, resample_strategy, scaler)

            if best_model_result:
                # Collect the feature importances for this dataset
                feature_importances = best_model_result.get('feature_importance', None)
                best_model_result['feature_importance'] = feature_importances

                datasets.append((dataset_number, best_model_result))
        except Exception as e:
            print(f"Error processing dataset {dataset_number}: {e}")

    return datasets



In [30]:
# Loading results, aggregating and saving them

def load_results(directory):
    result_files = [f for f in os.listdir(directory) if f.endswith('.csv') and 'standardize_False' not in f and 'SOSA' in f]
    all_results = []

    for file in result_files:
        filepath = os.path.join(directory, file)
        df = pd.read_csv(filepath)
        df['filename'] = file
        all_results.append(df)
    print (all_results)
    combined_results = pd.concat(all_results, ignore_index=True)
    # print (combined_results)
    return combined_results

def extract_preprocessing_parameters(filename):
    parts = filename.strip('.csv').split('_')
    impute_strategy = parts[3]
    transformation = parts[4]
    resample_strategy = parts[5]
    scaler = parts[6] if parts[6] in ['standard', 'robust'] else 'None'
    return impute_strategy, transformation, resample_strategy, scaler

def aggregate_best_results(combined_results):
    grouped = combined_results.groupby('Dataset')
    top_configs = []

    for dataset, group in grouped:
        # Calculate the combined score as:
        # combined_score = 0.5 * recall + 0.3 * pr_auc + 0.2 * f1
        group['combined_score'] = (group['Sensitivity'] * 0.5) + (group['PR AUC'] * 0.3) + (group['F1 Score'] * 0.2)

        # Find the best row based on the combined score
        best_row = group.loc[group['Combined Score'].idxmax()]
        impute_strategy, transformation, resample_strategy, scaler = extract_preprocessing_parameters(best_row['filename'])
        top_configs.append({
            'Dataset': dataset,
            'Best Model': best_row['Best Model'],
            'Combined Score': best_row['Combined Score'],
            'AUROC': best_row['AUROC'],
            'Specificity': best_row['Specificity'],
            'Sensitivity': best_row['Sensitivity'],
            'F1 Score': best_row['F1 Score'],
            'PR AUC': best_row['PR AUC'],
            'Best Parameters': best_row['Best Parameters'],
            'Transformation': transformation,
            'Resample Strategy': resample_strategy,
            'Impute Strategy': impute_strategy,
            'Scaler': scaler,
            'Feature Importance': best_row['Feature Importance']
        })

    return pd.DataFrame(top_configs)

def save_aggregated_results(aggregated_results, output_file):
    aggregated_results.to_csv(output_file, index=False)

# Directory containing the result files
directory = 'results\paramOptimization_and_strategies'

# Load results from all relevant files
combined_results = load_results(directory)

# Aggregate best results for each dataset
aggregated_results = aggregate_best_results(combined_results)

# Save aggregated results to a CSV file
output_file = 'aggregated_SOSA_results_v3.csv'
save_aggregated_results(aggregated_results, output_file)

# Display the aggregated results
# print(aggregated_results)

[   Dataset     Best Model  Combined Score     AUROC  Specificity  Sensitivity  \
0      171  Random Forest        0.607273  0.666667     0.777778          0.6   

   F1 Score    PR AUC                                    Best Parameters  \
0       0.6  0.624242  {'n_estimators': 300, 'min_samples_split': 2, ...   

                                  Feature Importance  \
0  [0.00262089 0.00288258 0.0002141  0.00093506 0...   

                                            filename  
0  SOSA_results_paramOptimization_KNN_ log_ADASYN...  ,    Dataset     Best Model  Combined Score     AUROC  Specificity  Sensitivity  \
0      171  Random Forest        0.607273  0.666667     0.777778          0.6   

   F1 Score    PR AUC                                    Best Parameters  \
0       0.6  0.624242  {'n_estimators': 300, 'min_samples_split': 2, ...   

                                  Feature Importance  \
0  [0.00262089 0.00288258 0.0002141  0.00093506 0...   

                              

In [23]:
# Work for datasets which yielded models with no available feature importances (part 2)

models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=5000),
    "Support Vector Classifier": SVC(random_state=42, probability=True),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}

all_results = []
impute_strategies = ['KNN', 'mean']
transformations = ['log', 'Yeo-Johnson']
resample_strategies = ['ADASYN', 'SMOTE']
scalers = ['standard', 'robust']
# Perform all combinations of transformations, resampling, imputation, and standardization
# and save each result to a file with an appropriate name
base_path = "datasets/SIMON-data/"
for impute_strategy in impute_strategies:
    for transformation in transformations:
        for resample_strategy in resample_strategies:
            for scaler in scalers:
                formatted_results = []
                print(f"\nProcessing with transformation={transformation}, resample={resample_strategy}, impute={impute_strategy}...")
                all_results = process_some_datasets(base_path, models, impute_strategy, transformation, resample_strategy, scaler)

                for dataset_number, result in all_results:
                    formatted_results.append({
                    "Dataset": dataset_number,
                    "Best Model": result['model'],
                    "Combined Score": result['combined_score'],
                    "AUROC": result['test_auc'],
                    "Specificity": result['specificity'],
                    "Sensitivity": result['sensitivity'],
                    "F1 Score": result['f1_score'],
                    "PR AUC": result['pr_auc'],
                    "Best Parameters": result['best_params'],
                    "Feature Importance": result['feature_importance']
                })

                # Save results to a CSV file using DataFrame
                results_df = pd.DataFrame(formatted_results)
                results_df.to_csv(f"results/paramOptimization_and_strategies/SOSA_results_paramOptimization_{impute_strategy}_ {transformation}_{resample_strategy}_{scaler}.csv", index=False)
     


Processing with transformation=log, resample=ADASYN, impute=KNN...
Processing Dataset 171...

Optimizing and training Random Forest...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None}
Best parameters for Random Forest: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None}

Optimizing and training Logistic Regression...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters: {'solver': 'saga', 'penalty': 'l2', 'C': 100.0}
Best parameters for Logistic Regression: {'solver': 'saga', 'penalty': 'l2', 'C': 100.0}
Logistic Regression discarded due to poor performance (AUROC=0.38).

Optimizing and training Support Vector Classifier...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters: {'kernel': 'linear', 'gamma': 'auto', 'C': 10.0}
Best parameters for

In [31]:
# Extract feature importance from the aggregated results
import pandas as pd
import ast
import numpy as np
import re
import pyarrow.feather as feather



# Path to the dataset directory (adjust as needed)
dataset_base_path = 'datasets/SIMON-data/'

# Load the results file containing the feature importances
results_df = pd.read_csv('aggregated_SOSA_results - Copy.csv')

def parse_feature_importance(feature_importance):
    """
    Parse the string representation of the feature importance into a numpy array.
    """
    # Clean the feature_importance string
    # if is nan, return None
    if pd.isnull(feature_importance):
        return None
    
    # if feature_importance is already a numpy array, return it as is
    if isinstance(feature_importance, np.ndarray):
        return feature_importance

    # second character might be a space, remove it
    if feature_importance[1] == ' ':
        feature_importance = feature_importance[0] + feature_importance[2:]

    # remove newlines and extra spaces using regular expression
    cleaned_feature_importance = re.sub(r'\s+', ' ', feature_importance.replace('\n', ' ').strip())  # Remove newlines, multiple spaces
    
    # replace '0. ' with '0 ' (removing unnecessary space after zeroes)
    cleaned_feature_importance = cleaned_feature_importance.replace('0. ', '0 ')
    
    # replace spaces with commas followed by a space
    cleaned_feature_importance = cleaned_feature_importance.replace(' ', ', ')

    # convert the cleaned string to a numpy array
    feature_importance = np.array(ast.literal_eval(cleaned_feature_importance))  # Convert to numpy array

    
    
    return feature_importance


def get_top_features(dataset_number, feature_importance):
    """
    Extract and return the top 5 most important features based on the feature importance values
    for the given dataset number. It also loads the feature names from the dataset.

    Args:
    - dataset_number (int): The dataset number for identification.
    - feature_importance (array): The feature importances for the given dataset.

    Returns:
    - top_features (list): The names of the top 5 features based on importance.
    - top_importance (list): The importance values of the top 5 features.
    """
    
    # Load the training data to get feature names
    dataset_path = os.path.join(dataset_base_path, str(dataset_number), 'data_training.feather')
    if not os.path.exists(dataset_path):
        print(f"Dataset {dataset_number} path does not exist: {dataset_path}")
        return None

    # Read the dataset (training data)
    data_training = pd.read_feather(dataset_path)
    
    # Remove the 'outcome' column to get the feature names
    feature_names = data_training.drop(columns=['outcome']).columns
    
    # Convert the feature_importance to a numpy array (in case it's not already)
    feature_importance = np.array(feature_importance)
    
    # Sort feature importance in descending order and get indices of top 5
    top_indices = np.argsort(feature_importance)[::-1][:5]

    # Get the top 5 feature importances
    top_importance = feature_importance[top_indices]
    
    # Get the corresponding top 5 features based on sorted indices
    top_features = feature_names[top_indices]
    
    # Print the results for debugging purposes
    print(f"Dataset {dataset_number}:")
    print("Top features:", top_features)
    print("Top importances:", top_importance)
    
    return top_features, top_importance


def get_top_features(dataset_number, feature_importance):
    """
    Extract and return the top 5 most important features based on the feature importance values
    for the given dataset number. It also loads the feature names from the dataset.

    Args:
    - dataset_number (int): The dataset number for identification.
    - feature_importance (str): The string representation of the feature importances for the given dataset.

    Returns:
    - top_features (list): The names of the top 5 features based on importance.
    - top_importance (list): The importance values of the top 5 features.
    """
    
    # if feature_importance is empty or nan, return None; can't call pd.isnull on a numpy array
    # if is not None, check if it is empty
    if feature_importance is None:
        # return two None values
        return None, None

    # Load the training data to get feature names
    dataset_path = os.path.join(dataset_base_path, str(dataset_number), 'data_training.feather')
    if not os.path.exists(dataset_path):
        print(f"Dataset {dataset_number} path does not exist: {dataset_path}")
        return None, None

    # Read the dataset (training data)
    data_training = pd.read_feather(dataset_path)
    
    # Remove the 'outcome' column to get the feature names
    feature_names = data_training.drop(columns=['outcome']).columns
        
    # Sort feature importance in descending order and get indices of top 5
    top_indices = np.argsort(feature_importance)[::-1][:5]

    # Get the top 5 feature importances
    top_importance = feature_importance[top_indices]
    
    # Get the corresponding top 5 features based on sorted indices
    top_features = feature_names[top_indices]

    # Top features: Index(['IFNa_CD8_pos_pSTAT5', 'Unstim_B_cell_pSTAT1', 'IFNg_CD8_pos_pSTAT5',
    #   'IFNa_Mono_pSTAT1', 'Unstim_CD8_pos_pSTAT5'],
    #  dtype='object')
    # Should contain only the feature names without the index and dtype
    top_features = [feature for feature in top_features]
    
    # Print the results for debugging purposes
    print(f"Dataset {dataset_number}:")
    print("Top features:", top_features)
    print("Top importances:", top_importance)
    
    return top_features, top_importance


total_results = []

for dataset_number, feature_importance in zip(results_df['Dataset'], results_df['Feature Importance']):
    print(f"Processing Dataset {dataset_number}...")
    print("Feature Importance:", feature_importance)
    print("After parsing:", parse_feature_importance(feature_importance))
    top_features, top_importance = get_top_features(dataset_number, parse_feature_importance(feature_importance))
    # append dataset number, top features, and top importance to the total_results list
    total_results.append({
        'Dataset': dataset_number,
        'Top Features': top_features,
        'Top Importance': top_importance
    })

# save the total_results to a DataFrame
total_results_df = pd.DataFrame(total_results)
total_results_df.to_csv('top_features_SOSA.csv', index=False)
    


Processing Dataset 1...
Feature Importance: [0.05339385 0.05877232 0.05125856 0.06134033 0.0367805  0.03089081
 0.03525284 0.03063579 0.03923972 0.02357208 0.03830949 0.04307175
 0.05501247 0.03147007 0.04386642 0.05090198 0.03687521 0.03792196
 0.05620535 0.04477627 0.03271109 0.04113058 0.04362362 0.02298692]
After parsing: [0.05339385 0.05877232 0.05125856 0.06134033 0.0367805  0.03089081
 0.03525284 0.03063579 0.03923972 0.02357208 0.03830949 0.04307175
 0.05501247 0.03147007 0.04386642 0.05090198 0.03687521 0.03792196
 0.05620535 0.04477627 0.03271109 0.04113058 0.04362362 0.02298692]
Dataset 1:
Top features: ['IFNg_CD4_pos_pSTAT1', 'IFNg_B_cell_pSTAT3', 'Unstim_CD8_pos_pSTAT1', 'Unstim_B_cell_pSTAT1', 'IFNg_B_cell_pSTAT1']
Top importances: [0.06134033 0.05877232 0.05620535 0.05501247 0.05339385]
Processing Dataset 3...
Feature Importance: [0.         0.         0.         0.         0.         0.0450379
 0.         0.         0.09853971 0.16899561 0.15908121 0.
 0.         0.    

In [111]:
# Using the big fluPRINT dataset: mining and transforming the data to our needs and findings

def process_big_dataset(impute_strategy='knn'):
    """
    Process the big fluPRINT dataset by filtering important features, pivoting, and imputing missing values.
    """
    big_dataset = pd.read_csv('datasets/fluprint/fluprint_export.csv')

    # extract from file, column "Top Features"
    important_features = pd.read_csv('top_features_SOSA.csv')['Top Features'].apply(ast.literal_eval).tolist()
    flattened_features = [item for sublist in important_features for item in sublist]
    important_features = flattened_features


    filtered_dataset = big_dataset[big_dataset['name_formatted'].isin(important_features)]
    filtered_dataset['outcome'] = filtered_dataset['vaccine_response'].apply(lambda x: 1 if x == 1 else 0)



    filtered_dataset['data'] = pd.to_numeric(filtered_dataset['data'], errors='coerce')


    pivoted_dataset = filtered_dataset.pivot_table(
        index=['donor_id', 'study_id', 'outcome'],  # Group by relevant identifiers
        columns='name_formatted',  # Feature names
        values='data',  # The actual measurements
        aggfunc='first'  # In case there are multiple values, take the first one (adjust if needed)
    )


    if impute_strategy == 'knn':
        knn_imputer = KNNImputer(n_neighbors=5)  # Set number of neighbors for KNN
        pivoted_dataset_imputed = pd.DataFrame(knn_imputer.fit_transform(pivoted_dataset), 
                                            columns=pivoted_dataset.columns,
                                            index=pivoted_dataset.index)
    elif impute_strategy == 'mean':
        pivoted_dataset_imputed = pivoted_dataset.fillna(pivoted_dataset.mean())
    elif impute_strategy == 'median':
        pivoted_dataset_imputed = pivoted_dataset.fillna(pivoted_dataset.median())
    else:
        # fill with zeroes
        pivoted_dataset_imputed = pivoted_dataset.fillna(0)

    print(pivoted_dataset_imputed.head())  # Preview imputed dataset


    # pivoted_dataset_imputed.to_csv('processed_big_dataset.csv')

    return pivoted_dataset_imputed

processed_big_dataset = process_big_dataset('median')
processed_big_dataset.to_csv('processed_big_dataset.csv')

name_formatted             CD161_pos_CD45RA_neg_Tregs  \
donor_id study_id outcome                               
1        30       0                              5.59   
2        30       0                              3.28   
3        30       0                              3.49   
4        30       0                              3.28   
5        30       0                              3.64   

name_formatted             CD161_pos_CD45RA_pos_Tregs  CD161_pos_NK_cells  \
donor_id study_id outcome                                                   
1        30       0                              0.00                75.5   
2        30       0                              0.21                64.8   
3        30       0                              0.30                40.7   
4        30       0                              0.21                64.8   
5        30       0                              0.11                71.7   

name_formatted             CD27_pos_CD8_pos_T_cells  CD57_po

C:\Users\Tibuman\AppData\Local\Temp\ipykernel_20272\1944890566.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_dataset['outcome'] = filtered_dataset['vaccine_response'].apply(lambda x: 1 if x == 1 else 0)
C:\Users\Tibuman\AppData\Local\Temp\ipykernel_20272\1944890566.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_dataset['data'] = pd.to_numeric(filtered_dataset['data'], errors='coerce')


In [14]:
# Training and evaluating on the big dataset

def calculate_sensitivity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tp / (tp + fn)

def train_random_forest(X_train, y_train, X_test, y_test, n_estimators=200, max_depth=20):
    rf_model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, max_features='sqrt', random_state=42)

    # resampling using ADASYN
    adasyn = ADASYN(sampling_strategy='minority', random_state=42)
    X_resampled, y_resampled = adasyn.fit_resample(X_train, y_train)

    # scaling using RobustScaler
    scaler = RobustScaler()
    X_rescaled = scaler.fit_transform(X_resampled)

    # convert back to DataFrame to retain feature names
    X_rescaled_df = pd.DataFrame(X_rescaled, columns=X_train.columns)

    # Train Random Forest
    rf_model.fit(X_rescaled_df, y_resampled)

    # predict and calculate metrics
    rf_pred = rf_model.predict(X_test)
    rf_prob = rf_model.predict_proba(X_test)[:, 1]

    rf_auc = roc_auc_score(y_test, rf_prob)
    rf_accuracy = accuracy_score(y_test, rf_pred)
    rf_sensitivity = calculate_sensitivity(y_test, rf_pred)
    rf_f1 = f1_score(y_test, rf_pred)
    precision, recall, _ = precision_recall_curve(y_test, rf_prob)
    rf_pr_auc = average_precision_score(y_test, rf_prob)

    # save model to disk
    filename = 'models/finalized_rf_model.sav'
    pickle.dump(rf_model, open(filename, 'wb'))

    print(f"Random Forest - AUROC: {rf_auc:.4f}, Accuracy: {rf_accuracy:.4f}, Sensitivity: {rf_sensitivity:.4f}, F1 Score: {rf_f1:.4f}, PR AUC: {rf_pr_auc:.4f}")

def train_svc(X_train, y_train, X_test, y_test, kernel='linear', C=1000, gamma='scale'):
    svc_model = SVC(kernel=kernel, C=C, gamma=gamma, random_state=42, probability=True)

    # resampling using ADASYN
    adasyn = ADASYN(sampling_strategy='minority', random_state=42)
    X_resampled, y_resampled = adasyn.fit_resample(X_train, y_train)

    # scaling using StandardScaler for SVC
    scaler = StandardScaler()
    X_rescaled = scaler.fit_transform(X_resampled)

    svc_model.fit(X_rescaled, y_resampled)

    # predict and calculate metrics
    svc_pred = svc_model.predict(X_test)
    svc_prob = svc_model.predict_proba(X_test)[:, 1]

    svc_auc = roc_auc_score(y_test, svc_prob)
    svc_accuracy = accuracy_score(y_test, svc_pred)
    svc_sensitivity = calculate_sensitivity(y_test, svc_pred)
    svc_f1 = f1_score(y_test, svc_pred)
    precision, recall, _ = precision_recall_curve(y_test, svc_prob)
    svc_pr_auc = average_precision_score(y_test, svc_prob)

    # save model to disk
    filename = 'models/finalized_svc_model.sav'
    pickle.dump(svc_model, open(filename, 'wb'))


    print(f"SVC - AUROC: {svc_auc:.4f}, Accuracy: {svc_accuracy:.4f}, Sensitivity: {svc_sensitivity:.4f}, F1 Score: {svc_f1:.4f}, PR AUC: {svc_pr_auc:.4f}")


def train_models(dataset_path):
    processed_big_dataset = pd.read_csv(dataset_path)
    X = processed_big_dataset.drop(columns=['outcome'])
    y = processed_big_dataset['outcome']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\nTraining Random Forest...")
    train_random_forest(X_train, y_train, X_test, y_test, n_estimators=300, max_depth=20)

    print("\nTraining Support Vector Classifier...")
    train_svc(X_train, y_train, X_test, y_test, kernel='linear', C=1000, gamma='scale')

processed_big_dataset = pd.read_csv('processed_big_dataset.csv')
X = processed_big_dataset.drop(columns=['outcome'])
y = processed_big_dataset['outcome']

# train_models(X, y)

# Predict, given a new dataset and path of trained models; load the trained models and predict
def predict_new_data(new_data_path, model_paths):
    new_data = pd.read_csv(new_data_path)
    print(new_data.head())
    X_new = new_data.drop(columns=['outcome'])
    y_new = new_data['outcome']
    # map 'outcome' to binary (0: low, 1: high)
    # y_new = y_new.map({'low': 0, 'high': 1})
    print(y_new)
    # models are saved in `.sav` format
    rf_model = pickle.load(open(model_paths['finalized_rf_model'], 'rb'))
    svc_model = pickle.load(open(model_paths['finalized_svc_model'], 'rb'))
    # make sure new data is in the same format as the training data: impute, scale, etc.
    # predict using the models
    rf_pred = rf_model.predict(X_new)
    svc_pred = svc_model.predict(X_new)
    # calculate metrics
    rf_accuracy = accuracy_score(y_new, rf_pred)
    svc_accuracy = accuracy_score(y_new, svc_pred)
    rf_sensitivity = calculate_sensitivity(y_new, rf_pred)
    svc_sensitivity = calculate_sensitivity(y_new, svc_pred)
    rf_f1 = f1_score(y_new, rf_pred)
    svc_f1 = f1_score(y_new, svc_pred)
    rf_prob = rf_model.predict_proba(X_new)[:, 1]
    svc_prob = svc_model.predict_proba(X_new)[:, 1]
    rf_auc = roc_auc_score(y_new, rf_prob)
    svc_auc = roc_auc_score(y_new, svc_prob)
    rf_pr_auc = average_precision_score(y_new, rf_prob)
    svc_pr_auc = average_precision_score(y_new, svc_prob)
    print(f"Random Forest - AUROC: {rf_auc:.4f}, Accuracy: {rf_accuracy:.4f}, Sensitivity: {rf_sensitivity:.4f}, F1 Score: {rf_f1:.4f}, PR AUC: {rf_pr_auc:.4f}")
    print(f"SVC - AUROC: {svc_auc:.4f}, Accuracy: {svc_accuracy:.4f}, Sensitivity: {svc_sensitivity:.4f}, F1 Score: {svc_f1:.4f}, PR AUC: {svc_pr_auc:.4f}")
    # save the predictions and probabilities to a file
    new_data['rf_prediction'] = rf_pred
    new_data['svc_prediction'] = svc_pred
    new_data['rf_probability'] = rf_prob
    new_data['svc_probability'] = svc_prob
    new_data.to_csv('new_data_predictions.csv', index=False)
    

In [18]:
# Flow

def get_random_rows(data, n=5):
    return data.sample(n)

def get_random_rows_from_dataset(dataset_path, n=5):
    data = pd.read_csv(dataset_path)
    # save get_random_rows(data, n) to new file
    random_rows = get_random_rows(data, n)
    random_rows.to_csv('random_rows.csv', index=False)

# console based menu:
# 1. Train models (RF & SVC) on given dataset path (enter path)
# 2. Process big dataset (enter impute_strategy)
# 3. Predict response given a new dataset (enter path to new dataset, path to trained models)
def flow():
    while True:
        print("\nMenu:")
        print("1. Train models (RF & SVC) on given dataset path")
        print("2. Process big dataset")
        print("3. Predict response given a new dataset")
        print("4. Exit")
        choice = input("Enter your choice: ")
        if choice == '1':
            dataset_path = input("Enter the path to the dataset: ")
            train_models(dataset_path)
        elif choice == '2':
            impute_strategy = input("Enter the impute strategy (knn, mean, median): ")
            process_big_dataset(impute_strategy)
        elif choice == '3':
            new_data_path = input("Enter the path to the new dataset: ")
            model_paths = {
                'finalized_rf_model': 'models/finalized_rf_model.sav',
                'finalized_svc_model': 'models/finalized_svc_model.sav'
            }
            predict_new_data(new_data_path, model_paths)
        elif choice == '4':
            break
        else:
            print("Invalid choice. Please try again.")

flow()


Menu:
1. Train models (RF & SVC) on given dataset path
2. Process big dataset
3. Predict response given a new dataset
4. Exit


   donor_id  study_id  outcome  CD161_pos_CD45RA_neg_Tregs  \
0         1        30        0                        5.59   
1       415        18        0                        3.28   
2       285        18        0                        3.28   
3        91        29        0                        3.28   
4       685        15        0                        3.28   

   CD161_pos_CD45RA_pos_Tregs  CD161_pos_NK_cells  CD27_pos_CD8_pos_T_cells  \
0                        0.00                75.5                     90.80   
1                        0.21                64.8                     86.75   
2                        0.21                64.8                     86.75   
3                        0.21                64.8                     86.75   
4                        0.21                64.8                     86.75   

   CD57_pos_CD4_pos_T_cells  CD85j_pos_CD8_pos_T_cells  CD94_pos_NK_cells  \
0                      4.24                      38.30               98.9  

c:\Users\Tibuman\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
c:\Users\Tibuman\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


Random Forest - AUROC: 0.5000, Accuracy: 0.9000, Sensitivity: 0.0000, F1 Score: 0.0000, PR AUC: 0.1214
SVC - AUROC: 0.5000, Accuracy: 0.1000, Sensitivity: 1.0000, F1 Score: 0.1818, PR AUC: 0.1000

Menu:
1. Train models (RF & SVC) on given dataset path
2. Process big dataset
3. Predict response given a new dataset
4. Exit
